# Cosmic Engine — Shot-Graph / Stargate Sequence (manifest-driven)

This notebook exercises the new architecture landed in commit `a671efd` on branch `claude/keen-maxwell-sAB8n`:

- `scene/` — ShotGraph + Shot + Transition + Palette manifest
- `render/core.py` — Renderer ABC + name registry
- `render/adapters.py` — wraps gas-giant, Kerr black hole, slit-scan tunnel, exotic physics
- `render/kerr.py` — Kerr ray-tracer with Page–Thorne disk, frame-dragging, Doppler beaming
- `render/io.py` — ACES filmic + sRGB tonemap, EXR/MP4 writers
- `director/continuity.py` — histogram-matched crossfade + smoothstep blend
- `director/graph.py` — replaces the old `ffmpeg -c copy` concat with overlapping transitions

**Runtime:** set the Colab runtime to **GPU** (T4 is fine). The fluid sim and Kerr renderer need it.

Pipeline this notebook drives end-to-end:

1. Install deps and clone the branch.
2. Run the 8-test CPU suite to verify the architecture imports cleanly.
3. Load `scene/examples/jupiter_to_stargate.json` (Jovian approach → Kerr anomaly → slit-scan tunnel).
4. Render it to MP4 via `GraphRunner` and display inline.
5. Tweak the manifest live (palette, Kerr spin, transition kind) and re-render.
6. Hand-author a fresh ShotGraph from scratch.

## 1. Install dependencies

In [ ]:
!pip -q install taichi 'imageio[ffmpeg,pyav]' av numpy
!apt-get -qq install -y ffmpeg > /dev/null
import taichi, imageio, av, numpy as np
print('taichi', taichi.__version__, '| imageio', imageio.__version__, '| av', av.__version__, '| numpy', np.__version__)

## 2. Clone the branch

Uses a shallow clone of the feature branch directly under `/content/cosmic_engine`. Re-running the cell pulls the latest commit on the same branch.

In [ ]:
import os, subprocess, sys

REPO_URL  = 'https://github.com/pmcray/cosmic_engine.git'
BRANCH    = 'claude/keen-maxwell-sAB8n'
REPO_DIR  = '/content/cosmic_engine'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

head = subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip()
print('HEAD:', head)

## 3. Smoke test the architecture (CPU only)

All eight tests should pass. They cover manifest round-trip, camera interpolation, the palette registry, ACES tone-mapping behaviour, the histogram-matched crossfade, the transition engine, the example manifest loader, and an end-to-end `GraphRunner` run on the trivial constant renderer. **No GPU touched yet.**

In [ ]:
!python -m tests.test_manifest

## 4. Inspect the example manifest

In [ ]:
from scene.manifest import load_manifest
from scene.palette import PALETTES

MANIFEST_PATH = 'scene/examples/jupiter_to_stargate.json'
graph = load_manifest(MANIFEST_PATH)

print(f'Title: {graph.title}')
print('Shots:')
for s in graph.shots:
    print(f'  {s.id:18s} renderer={s.renderer:18s} dur={s.duration_frames:>3d}f @ {s.fps}fps  res={s.resolution} palette={s.palette.name}')
print('Transitions:')
for t in graph.transitions:
    print(f'  {t.from_shot:18s} -> {t.to_shot:18s} kind={t.kind:10s} dur={t.duration_frames}f params={t.params}')
print('\nAvailable palettes:', sorted(PALETTES.keys()))

## 5. Fast-mode override

Full 1920×1080 × 336 frames takes several minutes per shot on a T4 and the first Kerr render also pays a one-off Taichi JIT cost. The cell below downscales the in-memory graph for a sanity render that finishes in roughly a minute. Set `FAST_MODE = False` for the full version.

In [ ]:
FAST_MODE = True

if FAST_MODE:
    for s in graph.shots:
        s.resolution = (640, 360)
        s.duration_frames = max(24, s.duration_frames // 4)
    for t in graph.transitions:
        t.duration_frames = max(6, t.duration_frames // 3)

print('Effective render plan:')
for s in graph.shots:
    print(f'  {s.id:18s} {s.duration_frames:>3d}f  {s.resolution}')
for t in graph.transitions:
    print(f'  {t.from_shot} -> {t.to_shot:18s} {t.kind:10s} {t.duration_frames}f')

## 6. Render the example manifest end-to-end

First run also JIT-compiles every Taichi kernel. Expect a one-time 20–40 s warm-up before frames start rolling.

In [ ]:
import time, os
import render.adapters  # registers gas_giant, kerr_black_hole, slitscan_tunnel, exotic_physics
from director.graph import GraphRunner

os.makedirs('outputs', exist_ok=True)
out_path = 'outputs/jupiter_to_stargate.mp4'

t0 = time.time()
GraphRunner(graph, out_path).run()
dt = time.time() - t0
print(f'wrote {out_path} ({os.path.getsize(out_path) / 1024:.1f} KiB) in {dt:.1f}s')

## 7. Inline playback

In [ ]:
from IPython.display import HTML
from base64 import b64encode

def show_mp4(path, width=720):
    data = open(path, 'rb').read()
    b64  = b64encode(data).decode()
    return HTML(
        f'<video width={width} controls autoplay loop muted playsinline>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'
    )

show_mp4(out_path)

## 8. Tweak the manifest live

The `graph` is just dataclasses — mutate it in memory and re-render. Here we swap to the JWST NIRCam palette and spin the Kerr black hole up to near-extremal `a/M = 0.99`.

In [ ]:
for s in graph.shots:
    s.palette.name = 'jwst_nircam'

kerr = graph.shot_by_id('kerr_anomaly')
kerr.params['spin']            = 0.99
kerr.params['inclination_deg'] = 87.0
kerr.params['disk_outer']      = 18.0

out_path2 = 'outputs/jupiter_to_stargate_jwst_spin99.mp4'
GraphRunner(graph, out_path2).run()
show_mp4(out_path2)

## 9. Hand-author a fresh ShotGraph

Build a two-shot sequence — near-extremal Kerr followed by a slit-scan tunnel — straight from Python, using the slit-scan family for the transition.

In [ ]:
from scene.manifest import Camera, PaletteRef, Shot, ShotGraph, Transition

my_graph = ShotGraph(
    title='kerr_into_stargate',
    shots=[
        Shot(
            id='kerr',
            renderer='kerr_black_hole',
            params={'spin': 0.95, 'inclination_deg': 85.0, 'disk_outer': 16.0, 'steps': 240},
            palette=PaletteRef(name='hubble_sii_ha_oiii'),
            duration_frames=48,
            fps=24,
            resolution=(640, 360),
            motion_hint='approach',
        ),
        Shot(
            id='tunnel',
            renderer='slitscan_tunnel',
            params={'time_scale': 0.06},
            palette=PaletteRef(name='trumbull_2001', intensity=1.2),
            duration_frames=64,
            fps=24,
            resolution=(640, 360),
            motion_hint='tunnel',
        ),
    ],
    transitions=[
        Transition(from_shot='kerr', to_shot='tunnel', kind='slitscan', duration_frames=18, params={'intensity': 1.4}),
    ],
)
my_graph.validate()

out_path3 = 'outputs/custom_kerr_into_stargate.mp4'
GraphRunner(my_graph, out_path3).run()
show_mp4(out_path3)

## 10. Where to go next

- **Bump up quality.** Set `FAST_MODE = False`, restart at cell 4, and re-run cells 4–7. Full 1920×1080 with the original frame counts.
- **Try a different transition.** Change `t.kind` to `crossfade`, `match_cut`, or `hard_cut` and re-render.
- **Add a renderer.** Implement a subclass of `render.core.Renderer`, decorate it with `@register_renderer('your_name')`, and reference it from a Shot. Saturn-class gas giant, neutron star, nebula volumetric — anything from the roadmap fits this slot.
- **Save the modified manifest.** `from scene.manifest import dump_manifest; dump_manifest(graph, 'my_shot_graph.json')` writes a reproducible manifest you can commit.
- **Wire in temporal-coherent AI.** A `TemporalDiffusionRenderer` that wraps another renderer's output as ControlNet conditioning is the next big roadmap item.